# Local Document Reader
## Notebook 01: Document Ingestion

### Objective
This notebook builds the first stage of Local Document Reader: converting a PDF into page-level, machine-readable records. Each record retains text, source filename, page number, character count, document metadata, and an empty-page flag.

### Why this matters
Retrieval and answer generation need traceable source text. Keeping page boundaries now lets later RAG answers cite the page that supported them.

### 1. Imports
PyMuPDF reads PDF pages; `Path` keeps paths portable; pandas provides a compact inspection table.

**Expected output:** `All libraries imported successfully!`

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.ingestion.pdf_loader import extract_pdf_text
print('All libraries imported successfully!')

All libraries imported successfully!


### 2. Configure project paths
Using paths relative to the project root avoids hard-coding a specific Mac username or folder.

In [2]:
DATA_DIR = PROJECT_ROOT / 'data'
SAMPLE_DOCS_DIR = DATA_DIR / 'sample_documents'
SAMPLE_DOCS_DIR.mkdir(parents=True, exist_ok=True)
print('Project Root:', PROJECT_ROOT)
print('Sample Documents:', SAMPLE_DOCS_DIR)

Project Root: /Users/apple/Downloads/Assessment
Sample Documents: /Users/apple/Downloads/Assessment/data/sample_documents


### 3. Extract and inspect a PDF
`extract_pdf_text` returns a list rather than one large string, so every text segment stays linked to its originating page. Blank or scanned pages are safely retained with `has_text=False`; OCR can be added later.

**Validation:** the summary should show one row per PDF page, and page numbers should start at 1.

In [3]:
def inspect_extraction(extracted_pages):
    return pd.DataFrame([
        {key: page[key] for key in ("page_number", "character_count", "has_text", "source_file")}
        for page in extracted_pages
    ])

pdf_files = sorted(SAMPLE_DOCS_DIR.glob("*.pdf"))
print("PDF documents found:", len(pdf_files))
if pdf_files:
    extracted_pages = extract_pdf_text(pdf_files[0])
    display(inspect_extraction(extracted_pages))
    print()
    print("First page preview:")
    print(extracted_pages[0]["text"][:1000])
else:
    print("Add a PDF to data/sample_documents and run this cell again.")

PDF documents found: 1


,page_number,character_count,has_text,source_file
0,1,200,True,client_requirements.pdf
1,2,109,True,client_requirements.pdf



First page preview:
Client Requirements
Project: School Management System
Frontend: React
Backend: Python FastAPI
Database: PostgreSQL
Deployment: AWS
Deliverables: Admin dashboard, attendance management, fee management.


### 4. Automated validation
Run the following from the project root to validate normal extraction, blank-page handling, missing files, and non-PDF rejection: `venv/bin/python -m pytest -q`.

## 9. Text Preprocessing
PDF text can contain inconsistent Unicode, excess whitespace, artificial line breaks, and words split by hyphenation. We normalize it before embedding, while retaining paragraph boundaries because they often carry meaning.

**Why:** clean, consistent text improves retrieval quality without losing source-page metadata.

In [4]:
from app.processing.text_processing import (
    clean_pages, clean_text, create_document_chunks, save_chunks,
    split_text_into_chunks, validate_chunks,
)

sample_text = """PROJECT REQUIREMENTS

The   client requires require-
ments from different departments."""
print(clean_text(sample_text))

PROJECT REQUIREMENTS

The client requires requirements from different departments.


**Expected output:** `requirements` appears as one word, repeated spaces are removed, and the blank line remains a paragraph break. The cleaner normalizes Unicode, fixes a hyphen immediately followed by a line break, and joins wrapped lines.

## 10. Apply preprocessing while preserving metadata
Cleaning must not sever the relationship between text and its PDF page. `clean_pages` copies every page record, changing only the text-derived fields.

In [5]:
cleaned_pages = clean_pages(extracted_pages)
comparison = pd.DataFrame([
    {'page_number': raw['page_number'], 'raw_characters': len(raw['text']), 'cleaned_characters': len(clean['text'])}
    for raw, clean in zip(extracted_pages, cleaned_pages)
])
display(comparison)
assert [page['page_number'] for page in cleaned_pages] == [page['page_number'] for page in extracted_pages]

,page_number,raw_characters,cleaned_characters
0,1,200,200
1,2,109,109


## 11. Boundary-aware chunking
Long documents are too large to pass to a model on every question. We use approximate 1,000-character chunks with 150 shared characters. The splitter prefers paragraphs, then lines, sentences, and finally whitespace, preserving context across chunks.

In [6]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
assert CHUNK_SIZE > 0 and 0 <= CHUNK_OVERLAP < CHUNK_SIZE
sample_chunks = split_text_into_chunks(' '.join(['A sentence about attendance management.'] * 20), 150, 30)
print('Chunks:', len(sample_chunks))
print('First chunk length:', len(sample_chunks[0]))
assert all(chunk.strip() and len(chunk) <= 150 for chunk in sample_chunks)

Chunks: 7
First chunk length: 119


**Expected output:** multiple non-empty chunks, each no longer than 150 characters in this deliberately small demonstration. The production configuration above uses 1,000/150.

## 12. Chunk metadata, validation, and persistence
Each chunk receives a unique ID, source document, original page number, position on that page, character count, and document metadata. This is what makes later citations possible.

In [7]:
source_document = pdf_files[0].name
document_chunks = create_document_chunks(cleaned_pages, source_document)
validate_chunks(document_chunks)
chunks_df = pd.DataFrame(document_chunks)
display(chunks_df[['chunk_id', 'page_number', 'chunk_index', 'character_count']])

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
output_path = save_chunks(document_chunks, OUTPUT_DIR / f'{Path(source_document).stem}_chunks.json')
print('All chunk validation tests passed!')
print('Chunks saved to:', output_path)

,chunk_id,page_number,chunk_index,character_count
0,client_requirements_page_1_chunk_1,1,1,200
1,client_requirements_page_2_chunk_1,2,1,109


All chunk validation tests passed!
Chunks saved to: /Users/apple/Downloads/Assessment/data/processed/client_requirements_chunks.json


### Validation summary
The assertions above ensure non-empty text and valid settings. `validate_chunks` additionally checks unique IDs, sources, positive page numbers, and accurate character counts. Run `venv/bin/python -m pytest -q` from the project root for automated edge-case tests.

Current limitation: chunking happens page by page, so a sentence spanning pages remains separate. Character counts are also an approximation of model tokens.